In [1]:
import torch
from torch import nn
from transformers import BartTokenizer, BartForConditionalGeneration, Trainer, TrainingArguments, BartConfig
from tqdm.notebook import tqdm
import pandas as pd
from datasets import Dataset, load_metric, DatasetDict
from torch.nn.init import xavier_uniform_
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
'''tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)
print ("device ",device)
model = model.to(device)'''
mapping = {"N":"negation", "E":"entity swap", "#":"number swap", 
           "A":"to antonym", "I":"no information", 
           "X":"mutual exclusion"}
mapping2 = {"N":0, "E":1, "#":2, 
           "A":3, "I":4, 
           "X":5, "Ans":6}
mapping3 = {0:"N", 1:"E", 2:"#", 
           3:"A", 4:"I", 
           5:"X", 6:"Ans"} 
import torch.nn as nn
class MTLModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        # Initialize BART model for generation
        self.bart = BartForConditionalGeneration.from_pretrained("facebook/bart-base")

        # Initialize classification heads
        self.classifier_1 = nn.Linear(config.d_model, 2)
        self.classifier_2 = nn.Linear(config.d_model, len(mapping2))

        self.dropout = nn.Dropout(0.2)
        self.criterion = nn.CrossEntropyLoss()

        self.initial_weights = [1, 0.2]  # Weights for generation and classification losses
        self.loss_weights = self.initial_weights.copy()
        self.optimizer = None
        self.scheduler = None
        self.init_params()

    def init_params(self):
        # Initialize weights for classification heads
        for m in [self.classifier_1, self.classifier_2]:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, input_ids, attention_mask, labels=None, 
            classification_input_ids_task_1=None, classification_attention_mask_task_1=None, 
            classification_labels_task_1=None, 
            classification_input_ids_task_2=None, classification_attention_mask_task_2=None, 
            classification_labels_task_2=None):
        # Generation outputs
        generation_outputs = self.bart(
            input_ids=input_ids, 
            attention_mask=attention_mask, 
            labels=labels, 
            return_dict=True,
            output_hidden_states=True
        )

        # Classification outputs for task 1
        classification_outputs_1 = self.bart(
            input_ids=classification_input_ids_task_1,
            attention_mask=classification_attention_mask_task_1,
            return_dict=True,
            output_hidden_states=True
        )

        # Classification outputs for task 2
        classification_outputs_2 = self.bart(
            input_ids=classification_input_ids_task_2,
            attention_mask=classification_attention_mask_task_2,
            return_dict=True,
            output_hidden_states=True
        )

        # Extract hidden states from the classification model
        hidden_states_1 = classification_outputs_1.decoder_hidden_states[-1].detach()
        hidden_states_2 = classification_outputs_2.decoder_hidden_states[-1].detach()

        # Get EOS token hidden states for both tasks
        eos_token_id = self.bart.config.eos_token_id
        eos_mask_1 = classification_input_ids_task_1.eq(eos_token_id).detach()
        eos_mask_2 = classification_input_ids_task_2.eq(eos_token_id).detach()

        eos_hidden_state_1 = hidden_states_1[eos_mask_1].view(hidden_states_1.size(0), -1, hidden_states_1.size(-1))[:, -1, :]
        eos_hidden_state_2 = hidden_states_2[eos_mask_2].view(hidden_states_2.size(0), -1, hidden_states_2.size(-1))[:, -1, :]

        eos_hidden_state_1 = self.dropout(eos_hidden_state_1.float())
        eos_hidden_state_2 = self.dropout(eos_hidden_state_2.float())

        # Classification heads for both tasks
        logits_1 = self.classifier_1(eos_hidden_state_1)
        logits_2 = self.classifier_2(eos_hidden_state_2)

        if self.training:
            # Ensure classification labels have the correct shape and type
            if classification_labels_task_1 is not None:
                classification_labels_task_1 = classification_labels_task_1.long()

            if classification_labels_task_2 is not None:
                classification_labels_task_2 = classification_labels_task_2.long()

            # Calculate losses
            loss_g = generation_outputs.loss
            loss_r1 = self.criterion(logits_1, classification_labels_task_1) if classification_labels_task_1 is not None else torch.tensor(0.0)
            loss_r2 = self.criterion(logits_2, classification_labels_task_2) if classification_labels_task_2 is not None else torch.tensor(0.0)

            weighted_loss_g = self.loss_weights[0] * loss_g
            weighted_loss_r1 = self.loss_weights[1] * loss_r1
            weighted_loss_r2 = self.loss_weights[1] * loss_r2

            total_loss = weighted_loss_g + weighted_loss_r1 + weighted_loss_r2

            return total_loss, logits_1, logits_2, loss_g

        # Modify input for generation based on classification outputs
        predicted_class_1 = logits_1.argmax(dim=-1).detach()
        predicted_class_2 = logits_2.argmax(dim=-1).detach()
        modified_decoder_input_ids = self.modify_decoder_input(input_ids, predicted_class_1, predicted_class_2)

        # Generate text using modified input
        generation_outputs = self.bart.generate(
            input_ids=modified_decoder_input_ids, 
            attention_mask=attention_mask
        )

        return generation_outputs, logits_1, logits_2


    def modify_decoder_input(self, input_ids, predicted_class_1, predicted_class_2):
        # Combine classification results to modify the input
        predicted_class_1_ids = predicted_class_1.unsqueeze(1)
        predicted_class_2_ids = predicted_class_2.unsqueeze(1)
        combined_class_ids = torch.cat([predicted_class_1_ids, predicted_class_2_ids], dim=1)
        modified_input_ids = torch.cat([combined_class_ids, input_ids[:, 2:]], dim=1)
        return modified_input_ids

    def update_loss_weights(self):
        # Gradually shift weights towards generation
        gen_weight = self.initial_weights[0] * 1.03
        class_weight = self.initial_weights[1] * 0.97
        self.loss_weights = [gen_weight, class_weight]

    def configure_optimizers(self, learning_rate=5e-5, weight_decay=0.01, adam_epsilon=1e-8, warmup_steps=0, total_steps=0):
        no_decay = ["bias", "LayerNorm.weight"]
        optimizer_grouped_parameters = [
            {
                "params": [
                    p for n, p in self.named_parameters()
                    if not any(nd in n for nd in no_decay)
                ],
                "weight_decay": weight_decay
            },
            {
                "params": [
                    p for n, p in self.named_parameters()
                    if any(nd in n for nd in no_decay)
                ],
                "weight_decay": 0.0
            }
        ]
        optimizer = AdamW(
            optimizer_grouped_parameters,
            lr=learning_rate,
            eps=adam_epsilon
        )
        self.optimizer = optimizer
        
        if warmup_steps > 0 and total_steps > 0:
            scheduler = get_linear_schedule_with_warmup(
                optimizer,
                num_warmup_steps=warmup_steps,
                num_training_steps=total_steps
            )
        else:
            scheduler = None
        self.scheduler = scheduler
        
        return optimizer, scheduler

    def optimizer_step(self, epoch, batch_idx):
        if self.optimizer is not None:
            self.optimizer.step()
            self.optimizer.zero_grad()
            if self.scheduler is not None:
                self.scheduler.step()

2024-08-20 12:16:47.106685: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-08-20 12:16:47.147998: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-08-20 12:16:47.903369: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
from transformers import BartForConditionalGeneration, BartConfig, BartTokenizer
from safetensors.torch import load_file
import os

# output_dir = './checkpoint-50000'
output_dir = './fine-tuned-bartMTL-salience'

# Load the configuration
config = BartConfig.from_pretrained(output_dir)

# Reconstruct the model
model = MTLModel(config)
model.bart.model.encoder.load_state_dict(load_file(os.path.join(output_dir, "encoder.safetensors")))
model.bart.model.decoder.load_state_dict(load_file(os.path.join(output_dir, "decoder.safetensors")))
model.bart.model.shared.load_state_dict(load_file(os.path.join(output_dir, "shared.safetensors")))
model.bart.lm_head.load_state_dict(load_file(os.path.join(output_dir, "lm_head.safetensors")))
model.classifier_1.load_state_dict(load_file(os.path.join(output_dir, "classifier_1.safetensors")))
model.classifier_2.load_state_dict(load_file(os.path.join(output_dir, "classifier_2.safetensors")))
# Load the tokenizer
tokenizer = BartTokenizer.from_pretrained(output_dir)

# Now you can use `model` and `tokenizer` as needed
model.to(device)

MTLModel(
  (bart): BartForConditionalGeneration(
    (model): BartModel(
      (shared): BartScaledWordEmbedding(50265, 768, padding_idx=1)
      (encoder): BartEncoder(
        (embed_tokens): BartScaledWordEmbedding(50265, 768, padding_idx=1)
        (embed_positions): BartLearnedPositionalEmbedding(1026, 768)
        (layers): ModuleList(
          (0-5): 6 x BartEncoderLayer(
            (self_attn): BartAttention(
              (k_proj): Linear(in_features=768, out_features=768, bias=True)
              (v_proj): Linear(in_features=768, out_features=768, bias=True)
              (q_proj): Linear(in_features=768, out_features=768, bias=True)
              (out_proj): Linear(in_features=768, out_features=768, bias=True)
            )
            (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (activation_fn): GELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072,

In [3]:
# Example test case
context = "Example context, a long way farther, etc"
question = "What"
sal = "Example salient sentence"
def answerability(context, question):
    st = f"""Context: {context}

    Question: {question}"""     
    return st

def mission(context, question, label, sal):
    if label != "Ans":
        st = f"""Context: {context}

        Question: {question}

        Issue: {mapping[label]}

        Salient Sentence: {sal}

        Task: Generate a new question based on the context and salient sentence that can be answered with the given context, given the issue."""
    else:
        st = question
    return st

def labelLess(context, question, sal, ab):
    if ab == 0:
        st = f"""Context: {context}

        Question: {question}

        Salient Sentence: {sal}
        """
    else:
        st = question
    return st

def evaluate_model(context, question, sal):
    # Move model to the correct device
    model.to(device)
    model.eval()
    
    # Task 1: Classification with answerability function
    input_str_task_1 = answerability(context, question)
    
    # Tokenize input for Task 1
    inputs_task_1 = tokenizer(input_str_task_1, max_length=512, truncation=True, padding='max_length', return_tensors='pt')
    inputs_task_1 = {key: val.to(device) for key, val in inputs_task_1.items()}
    
    # Predict for Task 1
    with torch.no_grad():
        outputs_task_1 = model(
            input_ids=inputs_task_1['input_ids'],
            attention_mask=inputs_task_1['attention_mask'],
            classification_input_ids_task_1=inputs_task_1['input_ids'],
            classification_attention_mask_task_1=inputs_task_1['attention_mask'],
            classification_labels_task_1=None,
            classification_input_ids_task_2=inputs_task_1['input_ids'],
            classification_attention_mask_task_2=inputs_task_1['attention_mask'],
            classification_labels_task_2=None
        )
        
        # Extract the classification logits and predicted label for Task 1
        logits_task_1 = outputs_task_1[1]  # Access logits_1 from output tuple
        predicted_label_task_1 = logits_task_1.argmax(dim=-1).item()
    
    # Task 2: Classification with labelLess function
    input_str_task_2 = labelLess(context, question, sal, predicted_label_task_1)  # Replace 'ab' with appropriate value if needed
    
    # Tokenize input for Task 2
    inputs_task_2 = tokenizer(input_str_task_2, max_length=512, truncation=True, padding='max_length', return_tensors='pt')
    inputs_task_2 = {key: val.to(device) for key, val in inputs_task_2.items()}
    
    # Predict for Task 2
    with torch.no_grad():
        outputs_task_2 = model(
            input_ids=inputs_task_2['input_ids'],
            attention_mask=inputs_task_2['attention_mask'],
            classification_input_ids_task_1=inputs_task_2['input_ids'],
            classification_attention_mask_task_1=inputs_task_2['attention_mask'],
            classification_labels_task_1=None,
            classification_input_ids_task_2=inputs_task_2['input_ids'],
            classification_attention_mask_task_2=inputs_task_2['attention_mask'],
            classification_labels_task_2=None
        )
        
        # Extract the classification logits and predicted label for Task 2
        logits_task_2 = outputs_task_2[2]  # Access logits_2 from output tuple
        predicted_label_task_2 = logits_task_2.argmax(dim=-1).item()
    
    # Task 3: Text generation with mission function
    label_str = mapping3[predicted_label_task_2]  # Use the output of Task 2
    mission_input = mission(context, question, label_str, sal)
    
    # Tokenize input for Task 3 (generation)
    generation_inputs = tokenizer(mission_input, max_length=512, truncation=True, padding='max_length', return_tensors='pt')
    generation_inputs = {key: val.to(device) for key, val in generation_inputs.items()}
    
    # Generate text for Task 3
    with torch.no_grad():
        generated_ids = model(
            input_ids=generation_inputs['input_ids'],
            attention_mask=generation_inputs['attention_mask'],
            classification_input_ids_task_1=inputs_task_1['input_ids'],
            classification_attention_mask_task_1=inputs_task_1['attention_mask'],
            classification_labels_task_1=torch.tensor([-1] * generation_inputs['input_ids'].size(0)).to(device),
            classification_input_ids_task_2=inputs_task_2['input_ids'],
            classification_attention_mask_task_2=inputs_task_2['attention_mask'],
            classification_labels_task_2=torch.tensor([-1] * generation_inputs['input_ids'].size(0)).to(device)
        )
        generated_ids = generated_ids[0] if isinstance(generated_ids, tuple) else generated_ids
        generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    
    return predicted_label_task_1, mapping3[predicted_label_task_2], generated_text

print(evaluate_model(context, question, sal))


/home/hmoradis/.conda/envs/QA_env/lib/python3.11/site-packages/transformers/generation/utils.py:1258: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


(0, 'I', 'What is the minimum number of sentences a person can have to complete the book?')


In [4]:
df_test= pd.read_csv('dev_sal.csv')

def write(res_eval, fname):
    with open(fname, 'w') as f:
        for line in res_eval:
            f.write(f"{line}\n")
        f.close()
        
input_seq_eval = []
anses = []
classes = []
# labels_seq_eval = []
for index, row in tqdm(df_test.iterrows(), total=len(df_test)):
    #label = row['label']
    context = row['context']
    question = row['original_question']
    sal = row['answer_sentence']
    ans, class_s, input_s = evaluate_model(context, question, sal)
    anses.append(ans)
    classes.append(class_s)
    input_seq_eval.append(input_s)
    write(anses, 'answers.txt')
    write(input_seq_eval, 'bart_epoch3.txt')
    write(classes, 'bart_epoch3_labels.txt')
    #break

df_test["BART_ANS"] = input_seq_eval
df_test["BART_LABEL"] = classes
df_test.to_csv("after_3_epochs.csv")
df_test.to_csv("MRC Eval/after_3_epochs.csv")

  0%|          | 0/5945 [00:00<?, ?it/s]

/home/hmoradis/.conda/envs/QA_env/lib/python3.11/site-packages/transformers/generation/utils.py:1258: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


In [5]:
!mv bart_epoch3_labels.txt "MRC Eval/"
!mv bart_epoch3.txt "MRC Eval/"

In [6]:
df_test.to_csv("MRC Eval/SG-Net/after_3_epochs.csv")

In [7]:
df_test